In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import copy
import cobra.test

In [2]:
test_model = cobra.test.create_test_model(model_name='ecoli')
x = test_model.optimize().to_frame()
x[x.fluxes > 0.02].tail()

,fluxes,reduced_costs
UDCPDP,0.027298,0.000000e+00
UGMDDS,0.027298,-5.551115e-17
UHGADA,0.038226,5.551115e-17
UMPK,0.371375,0.000000e+00
VPAMTr,0.415702,0.000000e+00


In [3]:
# set objective to r1 and optimize without coupling
test_model = cobra.test.create_test_model(model_name='ecoli')
r1 = test_model.reactions.VPAMTr
r2 = test_model.reactions.TKT2
test_model.objective = {r1: 1}

test_model.optimize().to_frame().loc[[r1.id, r2.id], 'fluxes']

VPAMTr    1.000000e+03
TKT2     -9.647455e-18
Name: fluxes, dtype: float64

In [4]:
# constraint reaction 1 s.t. reaction 1 = c*reaction 2 or c = r1/r2
proxy_metabolite = cobra.Metabolite('proxy_metabolite')

res = pd.DataFrame(columns = ['r1_flux', 'r2_flux', 'coupling_coefficient'])
counter = 0
for cc in tqdm(np.arange(0.5,2.01,0.5)):
    test_model = cobra.test.create_test_model(model_name='ecoli')
    r1 = test_model.reactions.UHGADA
    r2 = test_model.reactions.USHD
    test_model.objective = {r1: 1}
    
    r1.add_metabolites({proxy_metabolite: 1})
    r2.add_metabolites({proxy_metabolite: -cc})
    
    x = test_model.optimize().to_frame()
    res.loc[counter, :] = [x.loc[r1.id, 'fluxes'], x.loc[r2.id, 'fluxes'], cc]
    counter += 1
    
    
res['r1/r2'] = res.r1_flux/res.r2_flux
res

100%|██████████| 4/4 [00:21<00:00,  5.39s/it]


,r1_flux,r2_flux,coupling_coefficient,r1/r2
0,1.16868e-31,0,0.5,inf
1,1.16868e-31,0,1,inf
2,9.04884e-32,6.03256e-32,1.5,1.5
3,1.10985,0.554924,2,2
